### Setup

In [20]:
import pandas as pd
import os
import re
from mistralai.client import Mistral
from rapidfuzz import fuzz, process
from tqdm import tqdm
from dotenv import load_dotenv
import json

In [21]:
load_dotenv(dotenv_path='../.env')

True

In [22]:
client = Mistral(api_key=os.getenv('MISTRAL_API_KEY'))

In [23]:
TEMPLATE_OLD_PATH = "../data/submission_template.csv"
TEMPLATE_PATH = "../data/submission_template_v3.csv"

In [24]:
template_df = pd.read_csv(TEMPLATE_PATH)
template_old_df = pd.read_csv(TEMPLATE_OLD_PATH)
template_df['doc_id'] = template_old_df['doc_id']
template_df

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1
...,...,...,...,...
10048,party_list_34_11_53,ไทยพิทักษ์ธรรม,0,party_list_34_11
10049,party_list_34_11_54,ความหวังใหม่,0,party_list_34_11
10050,party_list_34_11_55,ไทยรวมไทย,0,party_list_34_11
10051,party_list_34_11_56,เพื่อบ้านเมือง,0,party_list_34_11


### EDA

In [25]:
template_df['party_name'].unique()

array(['ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
       'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
       'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
       'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
       'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
       'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
       'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
       'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
       'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ', nan,
       'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
       'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
       'สร้างชาติ', 'ใหม่', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
       'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
       'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
       'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธ

In [26]:
template_df.head()

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1


In [27]:
NaN_df = template_df[
    template_df['party_name'].isna()
]

NaN_df

,id,party_name,votes,doc_id
737,constituency_14_2_10,NaN,0,constituency_14_2
738,constituency_14_2_11,NaN,0,constituency_14_2
739,constituency_14_2_12,NaN,0,constituency_14_2
740,constituency_14_2_13,NaN,0,constituency_14_2
741,constituency_14_2_14,NaN,0,constituency_14_2
742,constituency_14_2_15,NaN,0,constituency_14_2
743,constituency_14_2_16,NaN,0,constituency_14_2
744,constituency_14_2_17,NaN,0,constituency_14_2


In [28]:
# 737 - 744 Correctly Null
template_df.iloc[737]

id            constituency_14_2_10
party_name                     NaN
votes                            0
doc_id           constituency_14_2
Name: 737, dtype: object

### Extraction

In [29]:
def thai_num_to_int(text: str) -> int:

    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # remove non-numeric prefixes
    text = re.sub(r"[^\d,]", " ", text)

    match = re.search(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    return int(match.group().replace(",", ""))

In [30]:
KNOWN_PARTIES = [
    'ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
    'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
    'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
    'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
    'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
    'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
    'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
    'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
    'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ',
    'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
    'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
    'สร้างชาติ', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
    'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
    'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
    'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธรรมใหม่', 'กรีน',
    'แผ่นดินธรรม', 'ประชาไทย', 'ประชาอาสาชาติ',
    'เครือข่ายชาวนาแห่งประเทศไทย', 'ไทยรวมไทย', 'พลังไทยรักชาติ', 'ใหม่'
]

def is_party(text, threshold=70):
    match = process.extractOne(text, KNOWN_PARTIES, scorer=fuzz.ratio)
    return match and match[1] >= threshold

In [31]:
def extract_party_score_dict(md: str) -> dict:
    result = {}

    for line in md.split('\n'):
        if '---' in line or 'รวมคะแนน' in line:
            continue

        parts = [p.strip() for p in line.split('|') if p.strip()]
        if len(parts) < 2:
            continue

        for i, p in enumerate(parts):
            match = process.extractOne(p, KNOWN_PARTIES, scorer=fuzz.ratio)
            if match and match[1] >= 60:
                canonical_party = match[0]
                for vote_raw in reversed(parts[i+1:]):
                    try:
                        vote = thai_num_to_int(vote_raw)
                        result[canonical_party] = vote
                        break
                    except Exception:
                        pass
                break

    return result

In [32]:
def extraction(path):
    try:
        if not os.path.exists(path):
            return {}
        filename = os.path.basename(path)
        with open(path, "rb") as f:
            uploaded = client.files.upload(
                file={"file_name": filename, "content": f},
                purpose="ocr"
            )
        signed_url = client.files.get_signed_url(file_id=uploaded.id)
        result = client.ocr.process(
            model="mistral-ocr-latest",
            document={"type": "document_url", "document_url": signed_url.url}
        )
        # Combine all pages markdown
        #print(result.pages[0].markdown)
        md = "".join(page.markdown for page in result.pages)
        return extract_party_score_dict(md)
    except Exception as e:
        print(e.__str__())
        return {}

### Inference

In [33]:
def assign_votes(df, vote_dict, threshold=60):
    vote_dict = {k: v for k, v in vote_dict.items() if k != 'รวมคะแนนทั้งสิ้น'}
    keys = list(vote_dict.keys())

    def get_vote(party_name):
        if pd.isna(party_name):
            return 0
        match = process.extractOne(
            party_name,
            keys,
            scorer=fuzz.token_set_ratio  # changed from fuzz.ratio
        )
        if match is None:
            return 0
        best_key, score, _ = match
        return vote_dict[best_key] if score >= threshold else 0

    df['votes'] = df['party_name'].apply(get_vote)
    return df

In [34]:
submission_df = template_df.copy()

In [35]:
PREFIX = "../pdf/"

doc_ids = submission_df['doc_id'].unique()

In [36]:
happen = {}

for doc_id in tqdm(doc_ids, desc="Processing", unit="doc"):
	try:
		vote_dict = extraction(PREFIX + doc_id + '.pdf')
		happen[doc_id] = vote_dict
		mask = submission_df["doc_id"] == doc_id
		submission_df.loc[mask] = assign_votes(
			submission_df.loc[mask].copy(),
			vote_dict,
			threshold=60
		)
	except Exception as e:
		print(f"{doc_id}: {str(e)}")

Processing:   0%|          | 0/300 [00:00<?, ?doc/s]

Processing:  76%|███████▋  | 229/300 [28:53<59:45, 50.50s/doc]

API error occurred: Status 520 Content-Type "text/html; charset=UTF-8". Body: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>mistral.ai | 520: Web server is returning an unknown error</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 cla

Processing: 100%|██████████| 300/300 [40:38<00:00,  8.13s/doc]


In [37]:
with open("../anormaly/party_list_10_24.json") as f:
    data = json.load(f)

for item in data:
    party = item["สังกัดพรรคการเมือง"].strip()
    vote_raw = item["คะแนน"].strip()
    try:
        vote = thai_num_to_int(vote_raw)
        match = process.extractOne(party, KNOWN_PARTIES, scorer=fuzz.ratio)
        if match and match[1] >= 60:
            canonical = match[0]
            mask = (submission_df["doc_id"] == "party_list_10_24") & (submission_df["party_name"] == canonical)
            submission_df.loc[mask, "votes"] = vote
    except Exception as e:
        print(f"Error: {party} - {e}")

In [38]:
submission_df

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,14813,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,14368,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,979,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,244,constituency_10_1
4,constituency_10_1_5,พลวัต,351,constituency_10_1
...,...,...,...,...
10048,party_list_34_11_53,ไทยพิทักษ์ธรรม,14,party_list_34_11
10049,party_list_34_11_54,ความหวังใหม่,41,party_list_34_11
10050,party_list_34_11_55,ไทยรวมไทย,29,party_list_34_11
10051,party_list_34_11_56,เพื่อบ้านเมือง,41,party_list_34_11


In [39]:
for key, values in happen.items():
	if values == {}:
		print(key)

party_list_10_24
party_list_20_4


In [44]:
output_df = submission_df.drop(['doc_id', 'party_name'], axis=1)

In [45]:
output_df

,id,votes
0,constituency_10_1_1,14813
1,constituency_10_1_2,14368
2,constituency_10_1_3,979
3,constituency_10_1_4,244
4,constituency_10_1_5,351
...,...,...
10048,party_list_34_11_53,14
10049,party_list_34_11_54,41
10050,party_list_34_11_55,29
10051,party_list_34_11_56,41


In [46]:
output_df.to_csv("submission4.csv", index=False)